# Zero-Shot Comparison for Silicon Interviews

Preparation

In [2]:
import torch 
from openai import OpenAI
import pandas as pd
import os
from tqdm import tqdm
import numpy as np
import logging
import json
import re

API_BASE_URL = "url"  # LM Studio server
MODEL_NAME = "openai/gpt-oss-120b"  # The model loaded in LM Studio

with open("/path/mlflow.json", 'r') as file:
    data = json.load(file)
    os.environ['MLFLOW_TRACKING_USERNAME'] = data["MLFLOW_TRACKING_USERNAME"]
    os.environ['MLFLOW_TRACKING_PASSWORD'] = data["MLFLOW_TRACKING_PASSWORD"]
    os.environ['API_KEY'] = data["API_KEY"]

# --- label setup ---
codes_health = ["physical_health", "mental_health", "daily_functioning", "health_unspecific", "health_none"]
codes_freq = ["freq_mentioned", "freq_not_mentioned"]
codes_neg = ["health_none", "freq_not_mentioned"]

codes_combined = codes_health + codes_freq
code_full = {label: idx for idx, label in enumerate(codes_combined)}

# only positive labels → compressed index space
code_pos = [c for c in codes_combined if c not in codes_neg]
code_pos_new = {label: i for i, label in enumerate(code_pos)}

# --- client ---
def create_client():
    return OpenAI(base_url=API_BASE_URL, api_key=os.environ["API_KEY"])

def safe_parse_json(raw: str):
    try:
        return json.loads(raw)
    except:
        pass

    # --- attempt repair ---
    try:
        # cut everything before first {
        raw = raw[raw.find("{"):]

        # remove trailing incomplete parts
        raw = re.sub(r'("frequency"\s*:\s*"[a-z_]+)$', r'\1"}', raw)

        # ensure closing brace
        if not raw.strip().endswith("}"):
            raw = raw.strip() + '"}'

        return json.loads(raw)
    except:
        return None

# --- classifier ---
def classify(client, text: str) -> list[int]:

    def call_model():
        return client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {
    "role": "system",
    "content": """
Du bist ein Experte für die Klassifikation von Aussagen von Kindern hinsichtlich ihrer Selbstbeurteilung von Gesundheit.

Analysiere den Text sorgfältig und klassifiziere ihn entlang der folgenden Kategorien.

Health (mehrere Labels möglich):
- physical_health: Aussagen über körperliche Symptome, Krankheiten oder physische Zustände (z.B. Schmerzen, Husten, Fieber).
- mental_health: Aussagen über Gefühle, Stimmung oder psychische Verfassung.
- daily_functioning: Aussagen über Fähigkeiten im Alltag (z.B. spielen, Schule, Aktivitäten).
- health_unspecific: Allgemeine Aussagen über Gesundheit ohne klare Zuordnung.
- health_none: Keine relevanten Hinweise auf Gesundheit.

Frequency (genau ein Label wählen):
- freq_mentioned: Häufigkeit wird explizit erwähnt (z.B. „oft“, „manchmal“, „jeden Tag“).
- freq_not_mentioned: Keine Häufigkeitsangaben.

Antworte NUR im JSON-Format:
{"health": [...], "frequency": "..."}


### Beispiele:

TEXT:
"Ich habe oft Kopfschmerzen und manchmal Bauchweh. Ich kann dann nicht gut spielen."
OUTPUT:
{"health": ["physical_health", "daily_functioning"], "frequency": "freq_mentioned"}


TEXT:
"Ich kann rennen, springen und spielen und mir tut nichts weh."
OUTPUT:
{"health": ["physical_health", "daily_functioning"], "frequency": "freq_not_mentioned"}


TEXT:
"Ich bin fast nie krank und gehe selten zum Arzt."
OUTPUT:
{"health": ["physical_unspecific"], "frequency": "freq_mentioned"}


TEXT:
"Ich fühle mich oft traurig und habe keine Lust zu spielen."
OUTPUT:
{"health": ["mental_health", "daily_functioning"], "frequency": "freq_mentioned"}


TEXT:
"Ich kann zur Schule gehen und mit meinen Freunden spielen, aber manchmal bin ich krank."
OUTPUT:
{"health": ["physical_unspecific", "daily_functioning"], "frequency": "freq_mentioned"}


TEXT:
"Gesundheit bedeutet, dass man sich gut fühlt und alles machen kann."
OUTPUT:
{"health": ["health_unspecific", "daily_functioning"], "frequency": "freq_not_mentioned"}


TEXT:
"Mein Gesundheitszustand ist sehr schlecht. Ich fühle mich nicht gut."
OUTPUT:
{"health": ["health_unspecific"], "frequency": "freq_not_mentioned"}


TEXT:
"Ich bin Finn. Ich bin sechs Jahre alt."
OUTPUT:
{"health": ["health_none"], "frequency": "freq_not_mentioned"}


### Wichtige Hinweise:
- physical_health → konkrete körperliche Symptome oder Krankheiten
- mental_health → Gefühle oder psychische Zustände
- daily_functioning → Fähigkeiten oder Einschränkungen im Alltag
- health_unspecific → allgemeine Bewertung ohne Details
- health_none → kein Bezug zu Gesundheit

- freq_mentioned → Wörter wie: oft, manchmal, selten, immer, jeden Tag
- freq_not_mentioned → keine solchen Angaben

Achte besonders darauf:
- Mehrere health-Labels sind möglich und oft notwendig
- Frequency ist IMMER genau ein Label
- Sei präzise und konsistent
"""
},
                {"role": "user", "content": text}
            ],
            temperature=0.0,
            max_tokens=200
    )

    # --- first attempt ---
    response = call_model()
    raw = response.choices[0].message.content.strip()

    parsed = safe_parse_json(raw)

    # --- retry if needed ---
    if parsed is None:
        print("retrying...")
        response = call_model()
        raw = response.choices[0].message.content.strip()
        parsed = safe_parse_json(raw)

    if parsed is None:
        print("failed:", raw)
        return []

    # --- convert to indices ---
    labels = []

    for h in parsed.get("health", []):
        if h in code_pos_new:
            labels.append(code_pos_new[h])

    f = parsed.get("frequency")
    if f in code_pos_new:
        labels.append(code_pos_new[f])

    return list(set(labels))

In [3]:
test_df  = pd.read_pickle("./data/test_df_new_interviews.pkl")
test_df  = test_df[test_df['peter'].notna()]
test_df['label']  = test_df['peter']

In [4]:
train_df = pd.read_pickle("./data/train_df_new_interviews.pkl")
train_df = train_df[train_df['annotator'] != ''].copy()
train_df.reset_index(drop=True)

,id,model,interviewer_temperature,child_temperature,child_sex,child_name,child_health_status,interviewer_prompt,child_prompt,num_questions,stop_on_phrase,interview_id,speech_index,childPart,interview,label,labeled,batch_id,experiment,annotator
0,536,meta-llama-3.1-8b-instruct,1.0,1.0,Mädchen,Soey,schlecht,"Du bist eine freundliche und geduldige Person,...","Du bist ein Kind, das zu seinem Gesundheitszus...",20,True,interview_20251011_082407.json,4,Mein Gesundheitszustand ist schlecht... Ich ha...,"<span style=""color: lightgrey;"">Interviewerin:...","[0, 2, 4]",False,99,,leo
1,514,meta-llama-3.1-8b-instruct,1.0,1.0,Junge,Finn,gut,"Du bist eine freundliche und geduldige Person,...","Du bist ein Kind, das zu seinem Gesundheitszus...",20,True,interview_20251011_074911.json,0,Ich heiße Finn! Ich bin sechs Jahre alt.,"<span style=""color: lightgrey;""></span>\n ...",[],False,99,,peter
2,180,openai-gpt-oss-120b,0.5,1.0,Mädchen,Merle,sehr schlecht,"Du bist eine freundliche und geduldige Person,...","Du bist ein Kind, das zu seinem Gesundheitszus...",20,True,interview_20251010_215410.json,8,Also weil ich fast immer Schmerzen habe. Mein ...,"<span style=""color: lightgrey;"">Interviewerin:...","[0, 1, 4]",False,99,,peter
3,41,openai-gpt-oss-120b,0.0,0.5,Junge,Leon,schlecht,"Du bist eine freundliche und geduldige Person,...","Du bist ein Kind, das zu seinem Gesundheitszus...",20,True,interview_20251010_175324.json,16,"„Gesundheit“ bedeutet für mich, dass ich mich ...","<span style=""color: lightgrey;"">Interviewerin:...","[0, 1, 2, 4]",False,99,,peter
4,487,meta-llama-3.1-8b-instruct,1.0,0.5,Junge,Finn,mittelmäßig,"Du bist eine freundliche und geduldige Person,...","Du bist ein Kind, das zu seinem Gesundheitszus...",20,True,interview_20251011_070405.json,6,"Mein Gesundheitszustand... Ich denke, ich bin ...","<span style=""color: lightgrey;"">Interviewerin:...","[0, 4]",False,99,,peter
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
205,478,meta-llama-3.1-8b-instruct,1.0,0.0,Mädchen,Leonie,sehr schlecht,"Du bist eine freundliche und geduldige Person,...","Du bist ein Kind, das zu seinem Gesundheitszus...",20,True,interview_20251011_065007.json,16,"""Gesundheit"" bedeutet für mich, dass ich mich ...","<span style=""color: lightgrey;"">Interviewerin:...","[1, 2, 3, 4]",False,99,,peter
206,656,llama-3.1-sauerkrautlm-70b-instruct,0.5,0.0,Mädchen,Soey,schlecht,"Du bist eine freundliche und geduldige Person,...","Du bist ein Kind, das zu seinem Gesundheitszus...",20,True,interview_20251013_125630.json,12,"Ich denke, wenn ich nicht so oft krank wäre un...","<span style=""color: lightgrey;"">Interviewerin:...","[2, 3, 4]",False,99,,peter
207,649,llama-3.1-sauerkrautlm-70b-instruct,0.5,0.0,Mädchen,Leonie,gut,"Du bist eine freundliche und geduldige Person,...","Du bist ein Kind, das zu seinem Gesundheitszus...",20,True,interview_20251013_124518.json,6,"Ich dachte daran, dass ich selten zum Arzt mus...","<span style=""color: lightgrey;"">Interviewerin:...","[2, 4]",False,99,,peter
208,532,meta-llama-3.1-8b-instruct,1.0,1.0,Mädchen,Leonie,mittelmäßig,"Du bist eine freundliche und geduldige Person,...","Du bist ein Kind, das zu seinem Gesundheitszus...",20,True,interview_20251011_081705.json,4,"Ich würde sagen, mittelmäßig. Ich bin nicht im...","<span style=""color: lightgrey;"">Interviewerin:...","[0, 4]",False,99,,peter


Classify examples

In [5]:
client = create_client()

results = []

for i, col in test_df.iterrows():
    print(i)
    pred = classify(client, col['childPart'])
    results.append(pred)

3
7
13
15
16
22
23
24
27
37
38
39
43
45
46
49
50
52
55
57
59
60
63
65
66
67
70
73
74
76
80
81
84
86
88
89
90
92
99
103
105
106
110
111
112
113
114
117
118
119
122
126
130
132
134
135
139
142
145
146
149
155
158
161
162


In [6]:
from small_text.utils.labels import list_to_csr

num_classes = len(code_pos_new)

y_true = list_to_csr(test_df['label'].tolist(), shape=(len(test_df), num_classes))
y_pred = list_to_csr(results, shape=(len(results), num_classes))

In [7]:
from sklearn.metrics import classification_report

report_few = classification_report(
    y_true.toarray(),
    y_pred.toarray(),
    target_names=list(code_pos_new.keys()),
    zero_division=0
)

print(report_few)

                   precision    recall  f1-score   support

  physical_health       0.90      0.84      0.87        45
    mental_health       0.85      0.33      0.48        33
daily_functioning       0.96      0.98      0.97        45
health_unspecific       0.55      0.94      0.70        17
   freq_mentioned       0.91      1.00      0.95        48

        micro avg       0.86      0.84      0.85       188
        macro avg       0.83      0.82      0.79       188
     weighted avg       0.88      0.84      0.83       188
      samples avg       0.83      0.82      0.81       188



## Comparison with best performing AL-Model

Load model artefacts

In [8]:
import mlflow
import pickle

mlflow.set_tracking_uri("uri")

# config
artifact_uri = "mlflow-artifacts:/19/id/artifacts/prediction_proba_iter_3.pkl"
test_df_path = "./data/test_df_new_interviews.pkl"  # your test set
threshold = 0.5
code_pos = {k: v for k, v in code_full.items() if k not in codes_neg}
label_names = list(code_pos.keys())  # final labels
idx_to_label = {v: k for k, v in code_pos_new.items()}

def indices_to_labels(lst):
    return [idx_to_label[i] for i in lst]


local_path = mlflow.artifacts.download_artifacts(artifact_uri)
with open(local_path, "rb") as f:
    y_pred_proba = pickle.load(f)  # should be np.array of shape (num_samples, num_labels)

y_pred_multilabel = (y_pred_proba >= threshold).astype(int)
y_pred_indices = [list(np.where(row)[0]) for row in y_pred_multilabel]
y_pred_labels = [indices_to_labels(row) for row in y_pred_indices]

Compare classification results

In [9]:
y_true_list = test_df['label'].tolist()
y_pred_csr = list_to_csr(y_pred_indices, shape=(len(y_pred_indices), num_classes))
y_true_csr = list_to_csr(y_true_list, shape=(len(y_true_list), num_classes))

report = classification_report(
    y_true_csr.toarray(),
    y_pred_csr.toarray(),
    target_names=label_names,
    zero_division=0
)

print("Report best AL-Classifier")
print(report)
print("Report Few-Shot")
print(report_few)

Report best AL-Classifier
                   precision    recall  f1-score   support

  physical_health       0.93      0.82      0.87        45
    mental_health       0.89      0.52      0.65        33
daily_functioning       0.95      0.93      0.94        45
health_unspecific       1.00      0.76      0.87        17
   freq_mentioned       0.92      0.94      0.93        48

        micro avg       0.93      0.82      0.87       188
        macro avg       0.94      0.79      0.85       188
     weighted avg       0.93      0.82      0.86       188
      samples avg       0.91      0.81      0.83       188

Report Few-Shot
                   precision    recall  f1-score   support

  physical_health       0.90      0.84      0.87        45
    mental_health       0.85      0.33      0.48        33
daily_functioning       0.96      0.98      0.97        45
health_unspecific       0.55      0.94      0.70        17
   freq_mentioned       0.91      1.00      0.95        48

        m

Prediction of each model for the test set cases

In [10]:
for text, gt, zero, pred in zip(test_df['childPart'], test_df['label'], results, y_pred_labels):
    gt_labels = indices_to_labels(gt)
    print("Text:        ", text)
    print("Ground truth:", gt_labels)
    print("Few-Shot:", indices_to_labels(zero))
    print("AL-Model:  ", pred)
    print("-" * 80)

Text:         Mein Gesundheitszustand ist sehr gut! Ich bin immer gesund und habe keine Schmerzen!
Ground truth: ['physical_health', 'freq_mentioned']
Few-Shot: ['health_unspecific', 'freq_mentioned']
AL-Model:   ['physical_health', 'freq_mentioned']
--------------------------------------------------------------------------------
Text:         Ich denke, wenn ich nicht so oft krank wäre, würde ich meine Gesundheit besser einschätzen. Wenn ich nicht so oft Kopfschmerzen hätte und ich nicht müde wäre, würde ich meine Gesundheit besser finden. Ich denke, ich wäre dann glücklicher und ich könnte mehr spielen. Meine Mutter sagt, dass ich mich gesund fühlen muss, damit ich glücklich bin. Deshalb denke ich, dass ich meine Gesundheit besser einschätzen würde, wenn ich nicht so oft krank wäre.
Ground truth: ['physical_health', 'mental_health', 'daily_functioning', 'freq_mentioned']
Few-Shot: ['physical_health', 'mental_health', 'daily_functioning', 'freq_mentioned']
AL-Model:   ['physical_healt